# CP3 · Notebook 02 — Setup

Verifica entorno + carga Moondream2 + test rápido. **Primera ejecución descarga ~3.7 GB**.

Tiempo: 5-10 min primera vez, <30 s siguientes.

In [ ]:
import sys, platform
print(f'Python {sys.version.split()[0]}  ({platform.system()} {platform.machine()})')
assert sys.version_info >= (3, 10), 'Necesitas Python 3.10+'

In [ ]:
import numpy as np, torch, transformers, einops, PIL
print(f'numpy {np.__version__}')
print(f'torch {torch.__version__}  (cuda: {torch.cuda.is_available()})')
print(f'transformers {transformers.__version__}')
print(f'einops {einops.__version__}')
print(f'Pillow {PIL.__version__}')

## 1. Verificar dataset

In [ ]:
from pathlib import Path

DATA = Path('../datasets/dashcam-curated')
categories = ['trivial', 'urbano_standard', 'edge_visual', 'edge_semantic', 'trampa', 'ambigua']
missing = []
total = 0
for cat in categories:
    cat_dir = DATA / cat
    if not cat_dir.exists():
        missing.append(cat); continue
    imgs = list(cat_dir.glob('*.jpg')) + list(cat_dir.glob('*.png'))
    total += len(imgs)
    print(f'  {cat:20s} {len(imgs)} imágenes')

print(f'\nTotal: {total} imágenes')
if missing or total < 12:
    print(f'\n❌ Dataset incompleto. Ejecuta: python scripts/download_dataset.py')
    print(f'   Si la categoría {missing} no existe, el script no llegó a ella.')
else:
    print('\n✅ Dataset OK')

## 2. Cargar Moondream2

**Primera ejecución**: descarga ~3.7 GB (10 min con conexión decente).
**Siguientes**: caché local, <30 s.

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'vikhyatk/moondream2'
# Pin revision para reproducibilidad — actualizar cuando se valide nueva versión
REVISION = '2024-08-26'

print(f'Cargando {MODEL_ID} (revision={REVISION})...')
print('  Primera vez puede tardar 5-10 min (descarga ~3.7 GB)')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.eval()
print(f'  cargado en {time.time()-t0:.1f}s')
print(f'  params: {sum(p.numel() for p in model.parameters())/1e9:.2f} B')

## 3. Test rápido sobre 1 imagen

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Primera imagen disponible
all_imgs = sorted((DATA).rglob('*.jpg')) + sorted((DATA).rglob('*.png'))
img_path = all_imgs[0]
img = Image.open(img_path).convert('RGB')
print(f'Imagen test: {img_path.relative_to(DATA)}  size={img.size}')

t0 = time.perf_counter()
enc_image = model.encode_image(img)
response = model.answer_question(enc_image, 'Describe this driving scene in one paragraph.', tokenizer)
elapsed = time.perf_counter() - t0

print(f'\nLatencia inferencia: {elapsed:.1f}s')
print(f'\nRespuesta del modelo:')
print(response)

plt.figure(figsize=(10, 5))
plt.imshow(img); plt.title(f'Test: {img_path.name}'); plt.axis('off')
plt.show()

## 4. Cierre

In [ ]:
if not missing and total >= 12:
    print('✅ Setup OK — listo para 03_descripcion_basica.ipynb')
else:
    print('❌ Setup incompleto — revisa warnings')